In [3]:
import sys

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os

In [4]:
DWH  = r"D:\Ahmed\DataCarft_Material\Python-End-to-End-ETL-Pipeline-Project\Data_model" + "\\"
OUT  = r"D:\Ahmed\DataCarft_Material\Python-End-to-End-ETL-Pipeline-Project\Visualization" + "\\"
os.makedirs(OUT, exist_ok=True)

In [5]:
fact        = pd.read_csv(DWH + "FACT_SALES.csv")
dim_store   = pd.read_csv(DWH + "DIM_STORE.csv")
dim_product = pd.read_csv(DWH + "DIM_PRODUCT.csv")
dim_date    = pd.read_csv(DWH + "DIM_DATE.csv")
dim_customer= pd.read_csv(DWH + "DIM_CUSTOMER.csv")
dim_staff   = pd.read_csv(DWH + "DIM_STAFF.csv")

In [ ]:
fact = (fact
        .merge(dim_store[["SK_store","store_name","store_state"]], on="SK_store", how="left")
        .merge(dim_product[["SK_product","product_name","category_name","brand_name"]], on="SK_product", how="left")
        .merge(dim_date[["SK_date","year","month","month_name"]], left_on="SK_order_date", right_on="SK_date", how="left")
        .merge(dim_customer[["SK_customer","state"]], on="SK_customer", how="left"))

In [7]:
# ── Global style ──────────────────────────────────────────────────────────────
sns.set_theme(style="darkgrid", palette="muted")
PALETTE = ["#4C72B0","#DD8452","#55A868","#C44E52","#8172B3","#937860"]
plt.rcParams.update({"figure.facecolor": "#F8F9FA", "axes.facecolor": "#F0F2F5",
                     "font.family": "DejaVu Sans", "axes.titlesize": 13,
                     "axes.titleweight": "bold"})
 
def save(fig, name):
    path = OUT + name
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print(f"[OK] Saved: {path}")
    plt.close(fig)

In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 1 — How much did we sell?
# ══════════════════════════════════════════════════════════════════════════════
 
# 1-A  Total revenue per store ------------------------------------------------
rev_store = (fact.groupby("store_name")["revenue_usd"]
               .sum().sort_values(ascending=False).reset_index())
 
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(rev_store["store_name"], rev_store["revenue_usd"],
               color=PALETTE[:len(rev_store)])
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e6:.1f}M"))
for bar, val in zip(bars, rev_store["revenue_usd"]):
    ax.text(val + 10_000, bar.get_y() + bar.get_height()/2,
            f"${val:,.0f}", va="center", fontsize=9)
ax.set_title("Total Revenue per Store")
ax.set_xlabel("Revenue (USD)")
ax.invert_yaxis()
fig.tight_layout()
save(fig, "1A_revenue_per_store.png")

[OK] Saved: D:\Ahmed\DataCarft_Material\Python-End-to-End-ETL-Pipeline-Project\Visualization\1A_revenue_per_store.png


In [9]:
# 1-B  Total revenue per month ------------------------------------------------
rev_month = (fact.groupby(["year","month","month_name"])["revenue_usd"]
               .sum().reset_index()
               .sort_values(["year","month"]))
rev_month["label"] = rev_month["month_name"].str[:3] + " " + rev_month["year"].astype(str)
 
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(rev_month["label"], rev_month["revenue_usd"],
        marker="o", linewidth=2, color=PALETTE[0])
ax.fill_between(rev_month["label"], rev_month["revenue_usd"], alpha=0.15, color=PALETTE[0])
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}K"))
ax.set_xticks(range(len(rev_month["label"])))
ax.set_xticklabels(rev_month["label"], rotation=45, ha="right", fontsize=8)
ax.set_title("Total Revenue per Month")
ax.set_ylabel("Revenue (USD)")
fig.tight_layout()
save(fig, "1B_revenue_per_month.png")

[OK] Saved: D:\Ahmed\DataCarft_Material\Python-End-to-End-ETL-Pipeline-Project\Visualization\1B_revenue_per_month.png


In [10]:
# 1-C  Top 10 products by revenue ---------------------------------------------
rev_prod = (fact.groupby("product_name")["revenue_usd"]
              .sum().sort_values(ascending=False).head(10).reset_index())
 
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(rev_prod["product_name"], rev_prod["revenue_usd"],
               color=sns.color_palette("Blues_d", len(rev_prod)))
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}K"))
for bar, val in zip(bars, rev_prod["revenue_usd"]):
    ax.text(val + 500, bar.get_y() + bar.get_height()/2,
            f"${val:,.0f}", va="center", fontsize=8)
ax.set_title("Top 10 Products by Revenue")
ax.set_xlabel("Revenue (USD)")
ax.invert_yaxis()
fig.tight_layout()
save(fig, "1C_top10_products.png")
 

[OK] Saved: D:\Ahmed\DataCarft_Material\Python-End-to-End-ETL-Pipeline-Project\Visualization\1C_top10_products.png


In [11]:
# SECTION 2 — How fast do we deliver?
# ══════════════════════════════════════════════════════════════════════════════
 
# Use orders source file for delivery metrics
orders_path = r"D:\Ahmed\DataCarft_Material\Python-End-to-End-ETL-Pipeline-Project\Transformation\staging_2" + "\\"
orders = pd.read_csv(orders_path + "orders.csv", parse_dates=["order_date","shipped_date"])
orders["days_to_ship"] = (orders["shipped_date"] - orders["order_date"]).dt.days
 
# 2-A  On-time vs late delivery -----------------------------------------------
delivery_counts = orders["delivery_status"].value_counts().reset_index()
delivery_counts.columns = ["status","count"]
 
colors_map = {"On Time": "#55A868", "Late": "#C44E52"}
colors = [colors_map.get(s, "#999") for s in delivery_counts["status"]]
 
fig, ax = plt.subplots(figsize=(6, 5))
wedges, texts, autotexts = ax.pie(
    delivery_counts["count"], labels=delivery_counts["status"],
    autopct="%1.1f%%", colors=colors, startangle=90,
    wedgeprops={"edgecolor":"white","linewidth":2})
for at in autotexts:
    at.set_fontsize(11); at.set_fontweight("bold")
ax.set_title("On-Time vs Late Deliveries")
fig.tight_layout()
save(fig, "2A_delivery_status.png")

[OK] Saved: D:\Ahmed\DataCarft_Material\Python-End-to-End-ETL-Pipeline-Project\Visualization\2A_delivery_status.png


In [12]:
# 2-B  Distribution of days to ship -------------------------------------------
shipped = orders.dropna(subset=["days_to_ship"])
 
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(shipped["days_to_ship"], bins=15, kde=True,
             color=PALETTE[0], ax=ax, edgecolor="white")
median_days = shipped["days_to_ship"].median()
ax.axvline(median_days, color="#C44E52", linestyle="--", linewidth=1.8,
           label=f"Median: {median_days:.0f} days")
ax.legend()
ax.set_title("Distribution of Days to Ship")
ax.set_xlabel("Days from Order to Shipment")
ax.set_ylabel("Number of Orders")
fig.tight_layout()
save(fig, "2B_days_to_ship.png")

[OK] Saved: D:\Ahmed\DataCarft_Material\Python-End-to-End-ETL-Pipeline-Project\Visualization\2B_days_to_ship.png


In [13]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 3 — Who are our customers?
# ══════════════════════════════════════════════════════════════════════════════
 
# 3-A  Customers per state ----------------------------------------------------
cust_state = (dim_customer["state"].value_counts()
                .reset_index().rename(columns={"index":"state","state":"count","count":"count"}))
# pandas v2 compat
cust_state = dim_customer["state"].value_counts().reset_index()
cust_state.columns = ["state","count"]
cust_state = cust_state.head(15)
 
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=cust_state, y="state", x="count",
            palette="Blues_d", ax=ax)
for p in ax.patches:
    ax.text(p.get_width() + 1, p.get_y() + p.get_height()/2,
            f"{int(p.get_width())}", va="center", fontsize=9)
ax.set_title("Top 15 States by Number of Customers")
ax.set_xlabel("Number of Customers")
ax.set_ylabel("State")
fig.tight_layout()
save(fig, "3A_customers_per_state.png")

C:\Users\asus\AppData\Local\Temp\ipykernel_28156\96451771.py:14: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=cust_state, y="state", x="count",


[OK] Saved: D:\Ahmed\DataCarft_Material\Python-End-to-End-ETL-Pipeline-Project\Visualization\3A_customers_per_state.png


In [14]:
# 3-B  Local vs non-local customers -------------------------------------------
locality_counts = (fact.groupby("locality")["SK_sale"]
                     .count().reset_index()
                     .rename(columns={"SK_sale":"orders"}))
 
fig, ax = plt.subplots(figsize=(6, 5))
colors_loc = ["#4C72B0","#DD8452"]
wedges, texts, autotexts = ax.pie(
    locality_counts["orders"], labels=locality_counts["locality"],
    autopct="%1.1f%%", colors=colors_loc, startangle=90,
    wedgeprops={"edgecolor":"white","linewidth":2})
for at in autotexts:
    at.set_fontsize(11); at.set_fontweight("bold")
ax.set_title("Local vs Non-Local Customers")
fig.tight_layout()
save(fig, "3B_local_vs_nonlocal.png")
 
print("\n[DONE] All 7 charts saved to:", OUT)# 3-B  Local vs non-local customers -------------------------------------------
locality_counts = (fact.groupby("locality")["SK_sale"]
                     .count().reset_index()
                     .rename(columns={"SK_sale":"orders"}))
 
fig, ax = plt.subplots(figsize=(6, 5))
colors_loc = ["#4C72B0","#DD8452"]
wedges, texts, autotexts = ax.pie(
    locality_counts["orders"], labels=locality_counts["locality"],
    autopct="%1.1f%%", colors=colors_loc, startangle=90,
    wedgeprops={"edgecolor":"white","linewidth":2})
for at in autotexts:
    at.set_fontsize(11); at.set_fontweight("bold")
ax.set_title("Local vs Non-Local Customers")
fig.tight_layout()
save(fig, "3B_local_vs_nonlocal.png")
 
print("\n[DONE] All 7 charts saved to:", OUT)

[OK] Saved: D:\Ahmed\DataCarft_Material\Python-End-to-End-ETL-Pipeline-Project\Visualization\3B_local_vs_nonlocal.png

[DONE] All 7 charts saved to: D:\Ahmed\DataCarft_Material\Python-End-to-End-ETL-Pipeline-Project\Visualization\
[OK] Saved: D:\Ahmed\DataCarft_Material\Python-End-to-End-ETL-Pipeline-Project\Visualization\3B_local_vs_nonlocal.png

[DONE] All 7 charts saved to: D:\Ahmed\DataCarft_Material\Python-End-to-End-ETL-Pipeline-Project\Visualization\
